# LLM 的“记忆力”：以 RoPE 的外推能力为切入点

谈到大语言模型的“记忆”，通常会区分三类：一是储存在模型参数中的**参数记忆**，二是由 attention 支撑的**上下文记忆**，三是以 RAG、数据库和工具调用为代表的**外部记忆**。相应地，人们也常用 LongBench、RULER、Needle-in-a-Haystack 等任务表现来评价模型的长上下文能力。

然而，这类评价更多是从人类使用工具的角度出发：模型能不能回答问题，能不能找到证据，能不能完成长文任务。它们衡量的是行为层面的有用性，而不是 Transformer 在机制层面究竟具备怎样的上下文能力。

如果从更低层的机制出发，问题就不再是 Transformer 能“记住多少字”，而是：

> 一个有限层数、有限宽度、带位置编码的注意力网络，在序列长度增长时，如何保持、路由、组合并利用上下文中的信息？

注意：此处的“路由”（routing）并不是 Transformer 原论文中的严格数学术语，而是近年来分析 Transformer 内部机制时常用的一种描述方式。它指的是：输入中某个 token 的信息如何经过 attention、残差连接、MLP 和多层表示变换，最终影响输出。换言之，路由关心的是信息在网络内部走了哪条路径。

### 上下文能力在不同层面下的细分

从这个角度看，Transformer 的上下文能力至少可以拆解为以下几个层面。

第一，**注意力可达性**。某个 token 的信息是否能够被后续 token 访问？这里的核心对象是 attention matrix。可以通过 attention entropy、attention mass、attention distance distribution 等指标观察模型是否真的在长距离位置之间建立了有效连接。

第二，**信息保持能力**。前文信息经过层层变换后，在 hidden state 中还剩下多少？这可以通过 mutual information、linear probing、representation similarity 等方法进行分析。如果远处信息虽然形式上仍在上下文窗口内，但在表示空间中已经不可恢复，那么它实际上已经被模型“遗忘”了。

第三，**位置编码的外推能力**。模型是否真的能处理超过训练长度的上下文？这涉及 RoPE、ALiBi、位置插值等位置编码机制的性质。对于 RoPE 来说，关键问题不是模型在某个长文本问答 benchmark 上得分多少，而是当相对距离超过训练窗口时，旋转相位、注意力分布和表示结构会发生怎样的变化。

第四，**信号-噪声比**。在长上下文中，目标 token 的信息是否会被大量无关 token 稀释？随着上下文长度 (n) 增大，softmax 归一化中的竞争项也随之增加。即使目标信息仍然存在，它在注意力分布中的有效权重也可能下降。

第五，**组合复杂度**。模型能否在多层 attention 中实现多步信息传递？单层 attention 更像一次全局读取，而多层 attention 才可能形成跨 token、跨层的路径组合。长上下文能力不仅取决于能不能读到远处信息，也取决于能不能把分散信息组合成新的表示。

第六，**有效上下文长度**。这不同于模型标称的 context window。真正有效的上下文长度，应该指某个远处信息源对输出分布仍然具有可测影响的最大距离。如果删除某个远处 token 后，输出分布几乎不变，那么这个 token 虽然位于窗口内，但实际上并没有被使用。

Transformer 的出色性能容易给人一种印象：只要把信息全部放进输入，attention 就会自动收集并利用这些信息，达到全量记忆的效果。但这更多是行为层面的直觉，并不等于机制层面的事实。事实上，即使在 prompt 中反复强调某条规则，模型仍可能在后续生成中违背它。不过，这也未必单纯是“记性差”的问题，不能用训练下属或提醒人类的心理学方式去理解 LLM。

其中，位置编码尤其关键。因为 attention 本身只计算 token 表示之间的相关性，而长上下文中的信息检索首先要求模型能够正确判断 token 之间的相对位置关系。如果位置关系在长度增长后发生失真，那么远处 token 虽然仍然在窗口内，却可能无法以训练阶段熟悉的方式参与 attention 计算。也就是说，所谓“记得住”，首先要求模型在更长序列上仍能维持稳定的位置几何结构。

因此，本文将**以 RoPE 为切入点，考察在固定长度 token block 上训练语言模型时，当模型遇到超过训练窗口的上下文，会发生什么**。重点包括：RoPE 数值外推行为的可视化、长距离位置关系下的 attention 变化，以及上下文 token 信息保持力度的分析。借此，我们尝试从机制层面重新理解大语言模型所谓的“记忆力”。

概括来说，长上下文能力可以分为三个层次：

**行为层指标**：LongBench、RULER、Needle-in-a-Haystack。
**机制层指标**：attention entropy、attention distance、representation retention、causal ablation、information flow。
**数理层问题**：复杂度、容量、位置外推、softmax 稀释、层数与组合路径。

本文关注的不是模型在长上下文任务上“好不好用”，而是 Transformer 在长度外推条件下，究竟如何保存、传递和利用信息。


## RoPE 的原理与实现

目前常见的位置编码方法包括绝对位置编码、相对位置编码、ALiBi 以及目前大多数大语言模型采用的 Rotary Position Embedding（RoPE）。

相比传统的位置编码，RoPE 最大的特点在于：它并不直接编码绝对位置，而是通过旋转变换，使 Attention 的计算天然依赖于 token 之间的相对距离。这一性质使得 RoPE 在长上下文任务中表现出更好的泛化能力，也使它成为研究 Transformer 上下文能力和长度外推问题的重要切入点。

因此，在讨论大语言模型的"记忆力"时，我们首先需要理解 RoPE 是如何工作的，以及它为什么既能支持长上下文，又会在超过训练长度后逐渐失效。

### RoPE 的数学原理

RoPE 的核心思想可以概括为一句话：

> **通过对 Query 和 Key 施加与位置相关的旋转变换，使 Attention 的内积结果仅依赖于 token 之间的相对位置，而非绝对位置。**

与传统的位置编码将位置向量直接加到 embedding 上不同，RoPE 将位置编码融入到了 Attention 的计算过程。


#### RoPE 的操作对象

设输入序列长度为 $L$，模型隐层维度为 $d_{\mathrm{model}}$。经过 embedding 和前面若干层 Transformer 之后，当前层的输入表示为

$$
X \in \mathbb{R}^{L \times d_{\mathrm{model}}}.
$$

对于某一个 attention head，Query 和 Key 由线性投影得到：

$$
Q = XW_Q,
\qquad
K = XW_K.
$$

其中

$$
W_Q,W_K \in \mathbb{R}^{d_{\mathrm{model}} \times d},
$$

$d$ 是每个 attention head 的维度。因此

$$
Q,K \in \mathbb{R}^{L \times d}.
$$

这时，$Q$ 的**第 $m$ 行**表示第 $m$ 个 token 对应的 Query 向量：

$$
q_m = Q[m,:] \in \mathbb{R}^{d}.
$$

$K$ 的**第 $n$ 行**表示第 $n$ 个 token 对应的 Key 向量：

$$
k_n = K[n,:] \in \mathbb{R}^{d}.
$$

也就是说，$q_m$ 不是一个独立的 Query 矩阵，而是整个 Query 矩阵 $Q$ 中对应第 $m$ 个位置的一行向量。类似地，$k_n$ 是 Key 矩阵 $K$ 中对应第 $n$ 个位置的一行向量。这里的 $d$ 指的是每个 attention head 内部的特征维度，我们默认每个 attention head 的维度 $d$ 为偶数。若 $d$ 为奇数，则只能对其中偶数个维度施加 RoPE，剩余维度保持不变，或者在模型设计时直接令 $d$ 为偶数。


#### 两两拆分

接下来，RoPE 把**同一个 Query 或 Key 向量内部的特征维度按相邻两维划分为若干二维子空间**（并不是把不同 token 两两分组，也不是把 Query 和 Key 两两分组）。

例如，对于 Query 向量

$$
q_m=(q_{m,0},q_{m,1},q_{m,2},q_{m,3},\cdots,q_{m,d-2},q_{m,d-1}),
$$

RoPE 将其划分为

$$
(q_{m,0},q_{m,1}),\quad
(q_{m,2},q_{m,3}),\quad
\cdots,\quad
(q_{m,d-2},q_{m,d-1}).
$$

对于 Key 向量

$$
k_n=(k_{n,0},k_{n,1},k_{n,2},k_{n,3},\cdots,k_{n,d-2},k_{n,d-1}),
$$

也同样划分为

$$
(k_{n,0},k_{n,1}),\quad
(k_{n,2},k_{n,3}),\quad
\cdots,\quad
(k_{n,d-2},k_{n,d-1}).
$$

也就是说，RoPE 对 Query 和 Key 都做旋转；区别只是 Query 使用它所在的位置 $m$，Key 使用它所在的位置 $n$。

对于第 $i$ 个二维子空间，定义对于 token 位置的角频率

$$
\theta_i = 10000^{-2i/d},
\qquad i=0,1,\cdots,\frac d2-1.
$$

在位置 $m$ 上，第 $i$ 个二维子空间对应的旋转矩阵为

$$
R_i(m)=
\begin{bmatrix}
\cos(m\theta_i) & -\sin(m\theta_i)\\
\sin(m\theta_i) & \cos(m\theta_i)
\end{bmatrix}.
$$

因此，Query 向量中第 $i$ 个二维子空间会被旋转为

$$
\begin{bmatrix}
\tilde q_{m,2i}\
\tilde q_{m,2i+1}
\end{bmatrix}
=
R_i(m)
\begin{bmatrix}
q_{m,2i}\
q_{m,2i+1}
\end{bmatrix}.
$$

同理，Key 向量中第 $i$ 个二维子空间会被旋转为

$$
\begin{bmatrix}
\tilde k_{n,2i}\
\tilde k_{n,2i+1}
\end{bmatrix}
=
R_i(n)
\begin{bmatrix}
k_{n,2i}\
k_{n,2i+1}
\end{bmatrix}.
$$

把所有二维子空间合在一起，就可以得到一个块对角矩阵：

$$
\mathcal R(m)
=
\operatorname{diag}
\left(
R_0(m),
R_1(m),
\cdots,
R_{d/2-1}(m)
\right).
$$

于是，RoPE 后的 Query 和 Key 可以写成

$$
\tilde q_m = \mathcal R(m)q_m,
\qquad
\tilde k_n = \mathcal R(n)k_n.
$$


#### RoPE 的应用

在 $q_m$ 与 $k_n$ 经 RoPE 处理后，Attention score 不再使用原始内积

$$
q_m^\top k_n,
$$

而是使用旋转后的内积

$$
\tilde q_m^\top \tilde k_n.
$$

展开可得

$$
\begin{aligned}
\tilde q_m^\top \tilde k_n
&=
\left(\mathcal R(m)q_m\right)^\top
\left(\mathcal R(n)k_n\right)\
&=
q_m^\top \mathcal R(m)^\top \mathcal R(n) k_n.
\end{aligned}
$$

由于二维旋转矩阵满足

$$
R_i(m)^\top R_i(n)=R_i(n-m),
$$

所以整个块对角旋转矩阵也满足

$$
\mathcal R(m)^\top \mathcal R(n)
=
\mathcal R(n-m).
$$

因此

$$
\tilde q_m^\top \tilde k_n
=
q_m^\top \mathcal R(n-m) k_n.
$$

对应地，加入 RoPE 之后，attention score 可以写成：

$$
\tilde s_{mn}
=
\frac{\tilde q_m^\top \tilde k_n}{\sqrt{d}}.
$$

将上面的结果代入，可得：

$$
\tilde s_{mn}
=
\frac{
q_m^\top \mathcal R(n-m) k_n
}{\sqrt{d}}.
$$

其中，$\tilde s_{mn}$ 表示位置 $m$ 的 Query 对位置 $n$ 的 Key 的注意力打分，$d$ 是每个 attention head 的维度。最后要注意的是，RoPE 通常只作用在 Query 和 Key 上，而不作用在 Value 上。因此，RoPE 改变的是“看哪里”的权重计算，而不是直接旋转被取出的内容 $v_n$。


#### 重点来了~

从上面的推导中我们可以发现 RoPE 最关键的性质：Query 和 Key 在进入 attention score 之前，确实分别使用了自己的绝对位置 $m$ 和 $n$。位置 $m$ 的 Query 会被旋转 $m\theta_i$，位置 $n$ 的 Key 会被旋转 $n\theta_i$。

但是二者做内积时，真正起作用的不是这两个绝对角度本身，而是它们之间的角度差：

$$
n\theta_i - m\theta_i = (n-m)\theta_i.
$$

也就是说，对于第 $i$ 个二维子空间，attention score 看到的不是“Query 在第 $m$ 个位置，Key 在第 $n$ 个位置”，而是：

$$
\text{Key 相对于 Query 偏移了 } n-m \text{ 个 token。}
$$

例如，如果当前位置是 $m=100$，被关注的 token 在 $n=90$，那么 RoPE 在内积中留下来的位置信息就是：

$$
n-m = -10.
$$

如果当前位置换成 $m=1000$，被关注的 token 在 $n=990$，那么：

$$
n-m = -10.
$$

二者的绝对位置完全不同，一个发生在 100 附近，一个发生在 1000 附近；但从 RoPE 的 attention score 看，它们具有相同的相对距离结构：Key 都在 Query 前面 10 个 token 的位置。

因此，RoPE 并不是直接告诉模型“这是第 100 个 token”或“这是第 1000 个 token”。它做的是：先用绝对位置决定 Query 和 Key 各自应该旋转多少角度，然后在二者内积时，让这两个绝对旋转角度相互抵消，只留下相对位移带来的相位差。

所以更准确地说：

> RoPE 使用绝对位置来旋转 Query 和 Key，attention score 最终感受到的是 Query 和 Key 之间的相对位置差。

从这个角度看，RoPE 的本质是：

> 在 Query 和 Key 的特征空间中引入与位置相关的旋转，使 token 之间的相对位置 自然融入 Attention score 里。

这也是 RoPE 比普通绝对位置编码更适合长上下文的原因之一：模型不必为每一个绝对位置单独学习一个位置表示，而是可以学习“相隔 1 个 token”“相隔 10 个 token”“相隔 100 个 token”这类相对距离模式。

需要注意的是，RoPE 本身并不会显式输出一个名为“相对距离”的变量。它只是通过旋转相位差，把相对位置信息编码进 Query 和 Key 的内积结构中。Transformer 会在训练过程中自己学习如何利用这种结构：某些 attention head 可能学会关注临近 token，某些 head 可能学会关注句首、段落边界或更远处的依赖关系。也就是说，相对位置信息并不是被人工规则直接读出，而是作为 attention score 中可被利用的几何结构，供模型在语言建模目标下自行解析和使用。

因此，更准确地说，RoPE 提供的是一种“相对位置可解析的表示空间”：它把相对距离编码进内积关系中，而 Transformer 通过训练学会从这种内积变化中提取有用的位置模式。


#### RoPE 的优势

这种设计带来了两个重要优势。

首先，由于位置关系体现在相对距离上，而不是固定的位置编号，因此模型在一定程度上具有比绝对位置编码更好的长度泛化能力。

其次，不同维度对应不同频率的旋转角速度。低频维度变化缓慢，可以稳定表示远距离的位置关系；高频维度变化较快，能够提供精细的局部位置信息。多种频率共同构成了一种类似傅里叶基底（Fourier basis）的表示，使模型能够同时感知局部和全局的位置关系。

然而，这种表示也埋下了长度外推的隐患。由于旋转角度会随着位置线性增长，当上下文长度远远超过训练窗口时，高频分量首先发生剧烈振荡，而低频分量也逐渐累积误差。虽然 RoPE 在数学上可以对任意长度计算旋转矩阵，但模型真正学习到的却只是训练窗口内的相位分布。因此，当推理长度不断增加时，Attention 中的位置关系会逐渐偏离训练阶段所学习到的分布，这正是长上下文性能下降的重要原因，也是后文将重点分析的内容。


### 为什么不直接编码相对位置？

既然 RoPE 最终也是为了让 attention score 感知相对距离 $n-m$，一个自然的问题是：为什么不直接把相对位置编码进去？

事实上，直接编码相对位置当然是可以的。很多位置编码方法正是这样做的。例如，可以在 attention score 中直接加入一个相对位置偏置：

$$
s_{mn}
=
\frac{q_m^\top k_n}{\sqrt{d}}
+
b_{n-m}.
$$

其中，$s_{mn}$ 表示位置 $m$ 的 Query 对位置 $n$ 的 Key 的注意力打分，$b_{n-m}$ 是一个只由相对距离 $n-m$ 决定的位置偏置。

这种方法非常直观。它相当于直接告诉模型：

> 当前位置 $m$ 和被关注位置 $n$ 之间相差 $n-m$ 个 token。

例如，如果 $n-m=-10$，就使用距离为 $-10$ 的位置偏置 $b_{-10}$；如果 $n-m=-100$，就使用距离为 $-100$ 的位置偏置 $b_{-100}$。

但是，这类直接相对位置编码通常有几个问题。

#### 第一，它往往是**内容无关的**

在下面的公式中：

$$
s_{mn}
=
\frac{q_m^\top k_n}{\sqrt{d}}
+
b_{n-m},
$$

相对位置项 $b_{n-m}$ 只和距离有关，而不和 token 的内容有关。也就是说，不管当前位置是名词、动词、括号还是变量名，只要相对距离相同，加入的偏置就是同一个。

这当然有用，但它更像是在 attention score 上加入一个固定的位置先验。例如，模型可以学到“附近的 token 通常更重要”，或者“某些距离上的 token 更值得关注”。但是，这种相对位置项并没有改变 Query 和 Key 之间的内容匹配方式。

RoPE 的做法不同。加入 RoPE 后，attention score 可以写成：

$$
\tilde s_{mn}
=
\frac{
q_m^\top \mathcal R(n-m) k_n
}{\sqrt{d}}.
$$

这里的相对距离 $n-m$ 不是作为一个额外的 bias 加到 attention score 上，而是直接参与了 Query 和 Key 的内积计算。

如果使用相对位置 bias，attention score 是：

$$
s_{mn}
=
\frac{q_m^\top k_n}{\sqrt d}
+
b_{n-m}.
$$

这表示：先计算 $q_m$ 和 $k_n$ 的内容相似度，再根据距离 $n-m$ 额外加上一个分数。这个位置项 $b_{n-m}$ 和 Query、Key 的具体内容无关。只要相对距离相同，加的分数就是一样的。

也就是说，bias 的作用更像是：

> 不管这两个 token 是什么内容，只要它们相隔这个距离，就统一加一点分或扣一点分。

RoPE 的方式不同。RoPE 后的 attention score 是：

$$
\tilde s_{mn}
=
\frac{
q_m^\top \mathcal R(n-m) k_n
}{\sqrt d}.
$$

这里的 $\mathcal R(n-m)$ 会先根据相对距离旋转 Key 的表示，然后再和 Query 做内积。也就是说，距离 $n-m$ 改变的不是一个额外分数，而是改变了 $q_m$ 和 $k_n$ 在向量空间中的对齐方式。

可以只看一个二维子空间。设

$$
q=(a,b),
\qquad
k=(c,d),
$$

相对距离对应的旋转角为

$$
\phi=(n-m)\theta_i.
$$

那么 RoPE 中这一组二维向量的内积为：

$$
q^\top R(\phi)k
=
(ac+bd)\cos\phi
+
(bc-ad)\sin\phi.
$$

可以看到，位置关系并不是简单地贡献一个固定偏置 $b_{n-m}$。相对距离 $\phi$ 会通过 $\cos\phi$ 和 $\sin\phi$ 改变不同内容分量的组合方式。所以，RoPE 并不是让位置本身变得“有内容”，而是让位置关系以一种乘法形式进入内容匹配过程。它把“两个 token 的内容是否匹配”和“它们相隔多远”放在同一个内积结构里共同计算。

换句话说，同样是距离 $-10$，不同的 Query 和 Key 会得到不同的影响；相同的相对距离 $n-m$，对不同的 $q_m$ 和 $k_n$ 可能产生不同影响；相同的一对 $q_m$ 和 $k_n$，在不同相对距离下也会得到不同的内积结果。

这种设计的潜在优势在于，**模型可以在训练中学习如何把不同类型的信息放到不同的频率子空间中**。某些 head 可以利用高频维度处理局部依赖，某些 head 可以利用低频维度处理较长距离关系。RoPE 提供的是一种可被模型利用的位置几何结构，而不是直接规定某个距离一定应该加多少分。

#### 第二，直接学习相对位置表会把长度外推问题暴露得更加明显。

假设我们为每个相对距离都学习一个位置偏置：

$$
b_{-1}, b_{-2}, b_{-3}, \cdots, b_{-L}.
$$

如果训练窗口长度为 $L$，那么模型只会学习到这个范围内的距离参数。一旦推理时上下文长度超过训练窗口，就会出现训练阶段没有对应参数的距离，例如：

$$
b_{-10000},\quad b_{-20000},\quad b_{-30000}.
$$

这时必须额外规定这些未见距离应该如何处理，例如截断、插值、分桶，或者让远距离共享同一个偏置。也就是说，直接学习相对位置表本身并没有自然定义训练范围之外的位置关系。

RoPE 的不同之处在于，它不是为每一个相对距离单独存一个参数，而是用连续函数生成位置关系：

$$
\cos((n-m)\theta_i),
\qquad
\sin((n-m)\theta_i).
$$

因此，从表示形式上说，任意相对距离 $n-m$ 都可以被代入计算。即使 $n-m$ 超过训练时见过的最大距离，RoPE 仍然能给出一个确定的旋转相位。

但这并不意味着 RoPE 真正解决了长度外推问题。它只保证了“未见距离在数学上可计算”，并不保证模型“学会了如何使用这些未见距离”。

这两者是不同层次的问题：

- **表示层面**：超过训练长度的位置关系是否有定义；
- **学习层面**：模型是否在训练中学会了如何利用这些位置关系。

RoPE 在表示层面比纯查表式相对位置编码更自然，因为它不需要为每一个距离单独学习参数，也不会在超过训练长度后立刻出现“没有这个位置参数”的问题。但是在学习层面，RoPE 仍然面对同样的分布外外推问题：模型训练时只见过有限范围内的相位组合，推理时遇到更长距离下的相位模式，仍然可能无法稳定解释。

因此，RoPE 的优势不应该被理解为“它能让模型自动学会任意长距离依赖”。更准确地说，RoPE 提供了一种连续、参数共享的位置几何结构，使得长度外推在形式上成为可能；但这种结构是否真的能被模型利用，仍然取决于训练长度、频率设计、模型规模和注意力头是否学到了可外推的位置规律。

#### 第三，RoPE 对 decoder-only language model 的推理实现比较友好。

在自回归生成中，模型会缓存过去 token 的 Key 和 Value，也就是 KV cache。使用 RoPE 时，过去位置 $n$ 的 Key 可以在生成时就被旋转好：

$$
\tilde k_n = \mathcal R(n)k_n.
$$

当模型生成到新位置 $m$ 时，只需要计算当前 Query 的旋转：

$$
\tilde q_m = \mathcal R(m)q_m.
$$

然后直接计算：

$$
\tilde q_m^\top \tilde k_n.
$$

这样，RoPE 和普通 attention 的矩阵乘法形式非常兼容，不需要在每一步显式构造复杂的相对位置表。